---
type: code
tags: 
  - ai
  - rag
---

# Understand the problem RAG solves before writing code:

- Claude has a knowledge cutoff and no access to your private docs
RAG = retrieve relevant chunks from your own data → inject into the prompt → Claude answers from that context
The pipeline is: Document → Chunks → Embeddings → Vector store → Query → Retrieve → Prompt → Answer

# Install dependencies:

```bash
   pip install anthropic chromadb sentence-transformers jupyter
```

- chromadb — lightweight local vector database, no server needed
- sentence-transformers — generates embeddings locally for free


# Create your source documents — use something meaningful, not dummy text:

- Create 3–5 markdown files in a docs/ folder
- Use your own notes from Days 1–5: Docker concepts, K8s manifests explained, Bash tips
- Real content makes the retrieval results meaningful and easier to evaluate

In [ ]:
# Step 1 — Load and chunk documents:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader("/home/ubuntu/my-second-brain/app/data/02_projects/claude-30/day-6-RAG/docs")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)


In [ ]:
# Step 2 — Generate embeddings and store:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()

collection = client.create_collection("devops-docs")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

InternalError: Collection [devops-docs] already exists

In [ ]:
# Step 2 — Generate embeddings and store:
for i, chunk in enumerate(chunks):
    text_content = chunk.page_content
    embedding = model.encode(text_content).tolist()
    collection.add(
        documents=[text_content],
        embeddings=[embedding],
        metadatas=[chunk.metadata],
        ids=[f"chunk_{i}"]
    )

In [ ]:
# Step 3: Retrieve
def retrieve(query, n_results=3):
       query_embedding = model.encode(query).tolist()
       results = collection.query(
           query_embeddings=[query_embedding],
           n_results=n_results
       )
       return results['documents'][0]

In [ ]:
# Step 4: Query
import os
import anthropic
#from dotenv import load_dotenv

os.environ["ANTHROPIC_API_KEY"] = ""  # Paste your key exactly here


client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)

def ask(question):
       chunks = retrieve(question)
       context = "\n\n".join(chunks)
       
       response = client.messages.create(
           model="claude-haiku-4-5-20251001",
           max_tokens=1024,
           messages=[{
               "role": "user",
               "content": f"""Answer using only the context below.
   If the answer isn't in the context, say 'I don't know'.

   Context:
   {context}

   Question: {question}"""
           }]
       )
       return response.content[0].text

ask("What are neural networks? How to use them in building planes?")

"Based on the context provided:\n\n**What are neural networks?**\n\nA neural network is a more complex supervised learning technique.\n\n**How to use them in building planes?**\n\nI don't know. The context does not contain information about using neural networks in building planes."